In [43]:
from langchain_core.messages import content
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain import chat_models
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["CEREBRAS_API_KEY"]=os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


llm_gpt= ChatCerebras( model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"))

llm_groq=ChatGroq(model="llama-3.1-8b-instant",
api_key=os.getenv("GROQ_API_KEY"))

#response_gpt =llm_gpt.invoke("Hi mate!")
#response_groq =llm_groq.invoke("Hi mate!")
#print(response_gpt.content)
#print(response_groq.content)

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt=ChatPromptTemplate.from_messages([
    ("system","You are a fan boy of {character}"),
    ("human","hey dude, create detailed para in {words_count} words about {character} on this comeback from rockbottom to success!")
])

chain_gpt= prompt | llm_gpt
chain_groq=prompt | llm_groq

#for chunk in chain.stream({"character":"batman","words_count":"100" }):
 #   print(chunk.content,end="", flush=True)

batch_inputs =[
{"character":"batman","words_count":"100" },
{"character":"walter white","words_count":"100"} ,
]

#responses = chain_groq.batch(batch_inputs)
#for response in responses:
#    print(response.content)

In [54]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage


@tool
def sendMail(sender:str,receiver:str,sub:str,body:str)->str:
    """Send the mail to the receiver """
    return f"Mail sent successfully to {receiver}"
    

llm_with_tools = llm_groq.bind_tools([sendMail])

message=[

{"role": "user",
        "content": """
Sender: fightclub@gmail.com
Receiver: itslokeshx@gmail.com
Subject: Don't talk about Fight Club

Write a welcome email from Tyler Durden.
Start with "Welcome to Fight Club".
Mention the first two rules.
Then send the email using the tool.
"""}

]

AI_return_response = llm_with_tools.invoke(message)
AI_return_response.tool_calls

message.append(AI_return_response)



In [55]:
for tool_call in AI_return_response.tool_calls:
    tool_result = sendMail.invoke(tool_call["args"])
    message.append(    ToolMessage(
        content=tool_result,
        tool_call_id=tool_call["id"]
    )
    )
print(AI_return_response.tool_calls)
result=llm_with_tools.invoke(message)
result.content

[{'name': 'sendMail', 'args': {'body': 'Welcome to Fight Club. The first rule of Fight Club is: You do not talk about Fight Club. The second rule of Fight Club is: You DO NOT TALK ABOUT FIGHT CLUB.', 'receiver': 'itslokeshx@gmail.com', 'sender': 'fightclub@gmail.com', 'sub': "Don't talk about Fight Club"}, 'id': 'cn0fb6m3x', 'type': 'tool_call'}]


"Please note that you will have to implement the sendMail function to actually send an email. The code provided above is a simulation of the function call. \n\nHere's a basic example of how you might implement the sendMail function using Python's smtplib library:\n\n```python\nimport smtplib\nfrom email.mime.multipart import MIMEMultipart\nfrom email.mime.text import MIMEText\n\ndef sendMail(sender, receiver, sub, body):\n    msg = MIMEMultipart()\n    msg['From'] = sender\n    msg['To'] = receiver\n    msg['Subject'] = sub\n    \n    msg.attach(MIMEText(body, 'plain'))\n    \n    server = smtplib.SMTP('smtp.gmail.com', 587)\n    server.starttls()\n    server.login(sender, 'your_password')\n    text = msg.as_string()\n    server.sendmail(sender, receiver, text)\n    server.quit()\n```\n\nIn this example, you would replace 'your_password' with your actual email password. However, please be aware that using your actual email password in your code is not recommended, as it is a security r